In [1]:
import requests
import re
import json
import pandas as pd
import time
from urllib.parse import urlparse
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

In [2]:
# 1. Obtener el HTML de la página del ranking
load_dotenv()
url = os.getenv("TARGET_URL_CPI")
BASE_URL = os.getenv("BASE_URL")

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36"
}

response = requests.get(url, headers=headers)
response.raise_for_status()  # Lanza error si la petición falla

In [3]:
# 2. Parsear HTML con BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")

In [4]:
# 3. Buscar la tabla principal y extraer los links
table = soup.find("table")
country_links = []


if table:
    rows = table.find_all("tr")
    for row in rows:
        cell = row.find("td")
        if cell and cell.a:
            relative_link = cell.a["href"]
            full_url = f"{BASE_URL}{relative_link}"
            country_links.append(full_url)

# Convertir la lista en DataFrame
df_links = pd.DataFrame(country_links, columns=["url"])

# Guardar como archivo Parquet
df_links.to_parquet("../../data/country_cpi_links.parquet", index=False)

In [5]:
# 4. Mostrar resultado
print("Links encontrados:", df_links.size)

Links encontrados: 182


In [6]:
# configuracion necesaria para descargar datos

master_df = pd.DataFrame()

pattern = r'arrayToDataTable\((\[\[.*?\]\])\)'

sleep_duration = 5

In [7]:
# Cargar los países válidos desde el archivo parquet
df_countries = pd.read_parquet("../../data/countries_with_hpi.parquet")
valid_countries = set(df_countries["country"].unique())

In [8]:
start_time = time.time()
for url in country_links:
    # Extraer país desde el URL
    country = urlparse(url).path.strip("/").split("/")[0]

    # Verificar si el país está en la lista válida
    if country not in valid_countries:
        print(f"⏭️ País {country} no está en la lista, se omite.")
        continue

    try:
        response = requests.get(url, headers=headers)
        html = response.text

        match = re.search(pattern, html, re.DOTALL)
        if match:
            data_str = match.group(1)
            data = json.loads(data_str)

            df = pd.DataFrame(data[1:], columns=data[0])
            df.columns = ["Period", "Value"]
            df["Country"] = country

            df_pivot = df.pivot(index="Period", columns="Country", values="Value")
            master_df = pd.merge(master_df, df_pivot, how="outer", left_index=True, right_index=True)

            print(f"✅ Datos extraídos correctamente para {country}")
        else:
            print(f"⚠️ No se encontraron datos en {url}")
    except Exception as e:
        print(f"❌ Error procesando {url}: {e}")
    
    time.sleep(sleep_duration)

end_time = time.time()
total_sleep_time = len(valid_countries) * sleep_duration
effective_time = end_time - start_time - total_sleep_time

print(f"⏳ Tiempo total (incluyendo esperas): {end_time - start_time:.2f} s")
print(f"😴 Tiempo en espera: {total_sleep_time:.2f} s")
print(f"⚡ Tiempo efectivo de ejecución: {effective_time:.2f} s")

⏭️ País Afghanistan no está en la lista, se omite.
⏭️ País Albania no está en la lista, se omite.
⏭️ País Algeria no está en la lista, se omite.
⏭️ País Andorra no está en la lista, se omite.
⏭️ País Angola no está en la lista, se omite.
⏭️ País Antigua-and-Barbuda no está en la lista, se omite.
⏭️ País Argentina no está en la lista, se omite.
⏭️ País Armenia no está en la lista, se omite.
⏭️ País Aruba no está en la lista, se omite.
✅ Datos extraídos correctamente para Australia
✅ Datos extraídos correctamente para Austria
⏭️ País Azerbaijan no está en la lista, se omite.
⏭️ País Bahamas no está en la lista, se omite.
⏭️ País Bahrain no está en la lista, se omite.
⏭️ País Bangladesh no está en la lista, se omite.
⏭️ País Barbados no está en la lista, se omite.
⏭️ País Belarus no está en la lista, se omite.
✅ Datos extraídos correctamente para Belgium
⏭️ País Belize no está en la lista, se omite.
⏭️ País Benin no está en la lista, se omite.
⏭️ País Bermuda no está en la lista, se omite

In [13]:
master_df.tail()

,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,Colombia,Croatia,...,South-Africa,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay
Period,,,,,,,,,,,,,,,,,,,,,
Sep 2020,NaN,108.67,108.21,5391.75,105.70,136.9,79.72,127.9984,105.29,103.18,...,116.8,100.74,103.69,108.06,100.44,102.18,182.90,260.280,109.1,106.6
Sep 2021,NaN,112.23,112.35,5944.21,109.92,142.9,83.98,128.4814,110.04,106.80,...,122.6,103.17,107.87,111.29,101.29,101.21,218.71,274.310,112.4,113.4
Sep 2022,NaN,124.58,125.90,6370.34,127.10,152.7,95.51,131.9948,122.63,120.21,...,106.1,108.82,117.55,122.70,104.53,107.70,401.10,296.808,122.3,124.0
Sep 2023,NaN,131.75,126.73,6700.66,135.18,158.5,100.39,132.1184,136.11,129.15,...,111.8,112.85,121.38,127.19,106.57,108.02,648.45,307.789,130.1,128.3
Sep 2024,NaN,134.13,132.21,6997.15,137.25,161.1,104.54,132.7618,144.02,133.11,...,116.1,114.65,123.42,128.76,107.50,108.68,969.11,315.301,133.5,133.5


In [14]:
# obtain paraguay url
template_url = country_links[0]
country = "Paraguay"

# Separar por "/"
parts = template_url.strip("/").split("/")

# Reemplazar el país (posición 3: después del dominio)

parts[3] = country

# Reconstruir la URL
custom_url = "/".join(parts) + "/"

In [15]:
# extract and include paraguay

response = requests.get(custom_url, headers=headers)
html = response.text

match = re.search(pattern, html, re.DOTALL)
if match:
    data_str = match.group(1)
    data = json.loads(data_str)

    df = pd.DataFrame(data[1:], columns=data[0])
    df.columns = ["Period", "Value"]
    df["Country"] = country

    df_pivot = df.pivot(index="Period", columns="Country", values="Value")
    master_df = pd.merge(master_df, df_pivot, how="outer", left_index=True, right_index=True)

    print(f"✅ Datos extraídos correctamente para {country}")

✅ Datos extraídos correctamente para Paraguay


In [17]:
unique_periods = pd.Series(master_df.index.astype(str)).unique()
print(unique_periods)

['Apr 1960' 'Apr 1961' 'Apr 1962' ... 'Sep 2022' 'Sep 2023' 'Sep 2024']


In [18]:
# Reset index to expose 'Period'
master_df2 = master_df.copy()
master_df2 = master_df.reset_index()

# Month to Quarter mapping
month_to_quarter = {
    "Jan": "Q1", "Feb": "Q1", "Mar": "Q1",
    "Apr": "Q2", "May": "Q2", "Jun": "Q2", "June": "Q2",
    "Jul": "Q3", "Aug": "Q3", "Sep": "Q3",
    "Oct": "Q4", "Nov": "Q4", "Dec": "Q4"
}

# Function to standardize each Period entry
def standardize_period(period):
    period = str(period).strip()

    # Match "Month Year" format
    match_month = re.match(r"^([A-Za-z]{3,4}) (\d{4})$", period)
    if match_month:
        month, year = match_month.groups()
        quarter = month_to_quarter.get(month)
        if quarter:
            return f"{quarter} {year}"

    # Match "Quarter Year" format
    match_quarter = re.match(r"^(Q[1-4]) (\d{4})$", period)
    if match_quarter:
        quarter, year = match_quarter.groups()
        return f"{quarter} {year}"

    # If no match, return None or original for inspection
    return None

In [19]:
# Apply the function
master_df2["Standardized_Period"] = master_df2["Period"].apply(standardize_period)

In [20]:
# Drop rows where standardization failed
master_df2 = master_df2[master_df2["Standardized_Period"].notna()].copy()

In [21]:
# Split standardized period into Quarter and Year
master_df2[["Quarter", "Year"]] = master_df2["Standardized_Period"].str.extract(r"(Q[1-4]) (\d{4})")
master_df2["Year"] = master_df2["Year"].astype(int)

In [22]:
# Reorder columns
cols = ["Quarter", "Year"] + [col for col in master_df2.columns if col not in ["Period", "Month", "Quarter", "Year", "Standardized_Period"]]
master_df2 = master_df2[cols]

In [23]:
# Sort by Year and Quarter
master_df2 = master_df2.sort_values(by=["Year", "Quarter"]).reset_index(drop=True)

In [24]:
master_df2.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay_x,Paraguay_y
0,Q1,1960,NaN,NaN,NaN,NaN,NaN,15.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.41,NaN,NaN,NaN
1,Q1,1960,NaN,NaN,NaN,NaN,NaN,15.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.37,NaN,NaN,NaN
2,Q1,1960,NaN,NaN,NaN,NaN,NaN,15.4,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.41,NaN,NaN,NaN
3,Q1,1960,7.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Q2,1960,NaN,NaN,NaN,NaN,NaN,15.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.54,NaN,NaN,NaN


In [26]:
master_df2["Quarter"].unique().tolist()

['Q1', 'Q2', 'Q3', 'Q4']

In [27]:
master_df2[master_df2["Quarter"].isnull()]

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay_x,Paraguay_y


In [28]:
def convert_quarter_to_int(df, quarter_col="Quarter"):
    """
    Converts quarter strings like 'Q1' to integers (1–4) in the specified column.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        quarter_col (str): The name of the column containing quarter strings.
    
    Returns:
        pd.DataFrame: A copy of the DataFrame with the quarter column converted to integers.
    """
    quarter_map = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
    df = df.copy()
    df[quarter_col] = df[quarter_col].map(quarter_map)
    return df

In [29]:
master_df3 = master_df2.copy()
master_df3 = convert_quarter_to_int(master_df3)
master_df3.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay_x,Paraguay_y
0,1,1960,NaN,NaN,NaN,NaN,NaN,15.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.41,NaN,NaN,NaN
1,1,1960,NaN,NaN,NaN,NaN,NaN,15.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.37,NaN,NaN,NaN
2,1,1960,NaN,NaN,NaN,NaN,NaN,15.4,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.41,NaN,NaN,NaN
3,1,1960,7.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,1960,NaN,NaN,NaN,NaN,NaN,15.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.54,NaN,NaN,NaN


In [30]:
master_df3["Quarter"].unique().tolist()

[1, 2, 3, 4]

In [31]:
def convert_year_to_int(df, year_col="Year"):
    """
    Converts the specified year column to integers, handling strings and other types.
    
    Parameters:
        df (pd.DataFrame): The input DataFrame.
        year_col (str): The name of the column containing year values.
    
    Returns:
        pd.DataFrame: A copy of the DataFrame with the year column as integers.
    """
    df = df.copy()
    df[year_col] = pd.to_numeric(df[year_col], errors="coerce").astype("Int64")
    return df

In [32]:
master_df3 = convert_year_to_int(master_df3)
master_df3.head()

,Quarter,Year,Australia,Austria,Belgium,Brazil,Bulgaria,Canada,Chile,China,...,South-Korea,Spain,Sweden,Switzerland,Thailand,Turkey,USA,United-Kingdom,Paraguay_x,Paraguay_y
0,1,1960,NaN,NaN,NaN,NaN,NaN,15.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.41,NaN,NaN,NaN
1,1,1960,NaN,NaN,NaN,NaN,NaN,15.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.37,NaN,NaN,NaN
2,1,1960,NaN,NaN,NaN,NaN,NaN,15.4,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.41,NaN,NaN,NaN
3,1,1960,7.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2,1960,NaN,NaN,NaN,NaN,NaN,15.5,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,29.54,NaN,NaN,NaN


In [33]:
master_df3["Year"].unique().tolist()

[1960,
 1961,
 1962,
 1963,
 1964,
 1965,
 1966,
 1967,
 1968,
 1969,
 1970,
 1971,
 1972,
 1973,
 1974,
 1975,
 1976,
 1977,
 1978,
 1979,
 1980,
 1981,
 1982,
 1983,
 1984,
 1985,
 1986,
 1987,
 1988,
 1989,
 1990,
 1991,
 1992,
 1993,
 1994,
 1995,
 1996,
 1997,
 1998,
 1999,
 2000,
 2001,
 2002,
 2003,
 2004,
 2005,
 2006,
 2007,
 2008,
 2009,
 2010,
 2011,
 2012,
 2013,
 2014,
 2015,
 2016,
 2017,
 2018,
 2019,
 2020,
 2021,
 2022,
 2023,
 2024,
 2025]

In [34]:
# Guardar como archivo Parquet
master_df3.to_parquet("../../data/cpi_values.parquet", index=False)